<a href="https://colab.research.google.com/github/41371112h/114-1/blob/main/%E8%A9%A6%E7%AE%97%E8%A1%A8%E5%9B%9E%E5%82%B3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# Cell 1：安裝 & 匯入
!pip install -q pandas gspread gspread_dataframe

import pandas as pd
import datetime


In [6]:
# Cell 2：匯入套件＆設定 Gemini API（使用 2.5 flash）

import pandas as pd
import datetime
import google.generativeai as genai

# 讀取 Colab 使用者金鑰（名稱：hw6）
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('hw6')
except ImportError:
    import os
    GEMINI_API_KEY = os.environ.get('hw6')

if GEMINI_API_KEY is None:
    raise ValueError("找不到名為 'hw6' 的 Gemini API key，請先在 Colab 設定。")

# 設定 API Key
genai.configure(api_key=GEMINI_API_KEY)

# ⭐ 最關鍵：使用可用的最新版模型
model = genai.GenerativeModel("gemini-2.5-flash")     # 建議用這個
# model = genai.GenerativeModel("gemini-2.5-flash-lite")   # 備用較省錢版

print("✅ Gemini 設定完成（使用 gemini-2.5-flash）")


一週課表預覽：


,日期,星期,課程名稱,時間(起),時間(迄),教室
0,2025/09/01,一,行銷管理,14:20,17:20,誠207
1,2025/09/01,一,網際網路程式設計,17:30,20:25,科技系TA509教室
2,2025/09/02,二,網際網路概論,9:10,12:10,科技系TB311教室
3,2025/09/02,二,統計學（一）,14:20,17:20,誠302
4,2025/09/03,三,手語,13:20,15:10,特114視聽室
5,2025/09/03,三,運算思維與程式設計,15:30,17:20,教401
6,2025/09/04,四,程式語言,9:10,12:10,科技系TB311教室
7,2025/09/04,四,組織行為,13:20,16:20,誠102
8,2025/09/05,五,體育（有氧舞蹈）,13:20,15:10,大韻律教室
9,2025/09/05,五,作業系統,17:30,20:25,科技系TA509教室


In [7]:
# Cell 3：產生 2025/09/01 ~ 2025/12/19 所有日期

start_date = "2025-09-01"
end_date   = "2025-12-19"

dates = pd.date_range(start=start_date, end=end_date, freq="D")
dates_df = pd.DataFrame({"日期": dates})

# 把 weekday 轉成「一、二、三、四、五、六、日」
weekday_map = {0: "一", 1: "二", 2: "三", 3: "四", 4: "五", 5: "六", 6: "日"}
dates_df["星期"] = dates_df["日期"].dt.weekday.map(weekday_map)

print("日期＋星期 預覽：")
dates_df.head(10)


日期＋星期 預覽：


,日期,星期
0,2025-09-01,一
1,2025-09-02,二
2,2025-09-03,三
3,2025-09-04,四
4,2025-09-05,五
5,2025-09-06,六
6,2025-09-07,日
7,2025-09-08,一
8,2025-09-09,二
9,2025-09-10,三


In [8]:
# Cell 4：展開整學期課表

# 確保一週課表的「星期」也是字串型態（例如 "一"、"二"...）
weekly_df["星期"] = weekly_df["星期"].astype(str)

# 我們只把 weekly_df 當作「模板」，不使用裡面的日期欄
# 所以先丟掉原本的「日期」，避免等等 merge 混淆
weekly_template = weekly_df.drop(columns=["日期"])

# 用「星期」來 merge：每一個日期會接上該星期的一週課表內容
full_df = dates_df.merge(weekly_template, on="星期", how="left")

# 只保留有課的那幾天（有些日子可能完全沒課，例如週六日）
full_df = full_df[full_df["課程名稱"].notna()].copy()

# 把日期改成字串格式：2025/09/01
full_df["日期"] = full_df["日期"].dt.strftime("%Y/%m/%d")

# 排序一下：先日期，再時間(起)
full_df = full_df.sort_values(by=["日期", "時間(起)"])

# 將欄位順序排成你要的樣子
full_df = full_df[["日期", "星期", "課程名稱", "時間(起)", "時間(迄)", "教室"]]

print("✅ 整學期課程紀錄（前 30 筆）：")
full_df.head(30)


✅ 整學期課程紀錄（前 30 筆）：


,日期,星期,課程名稱,時間(起),時間(迄),教室
0,2025/09/01,一,行銷管理,14:20,17:20,誠207
1,2025/09/01,一,網際網路程式設計,17:30,20:25,科技系TA509教室
3,2025/09/02,二,統計學（一）,14:20,17:20,誠302
2,2025/09/02,二,網際網路概論,9:10,12:10,科技系TB311教室
4,2025/09/03,三,手語,13:20,15:10,特114視聽室
5,2025/09/03,三,運算思維與程式設計,15:30,17:20,教401
7,2025/09/04,四,組織行為,13:20,16:20,誠102
6,2025/09/04,四,程式語言,9:10,12:10,科技系TB311教室
8,2025/09/05,五,體育（有氧舞蹈）,13:20,15:10,大韻律教室
9,2025/09/05,五,作業系統,17:30,20:25,科技系TA509教室


In [9]:
# Cell 5-1：Google 授權 & 連線
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
from gspread_dataframe import set_with_dataframe

creds, _ = default()
gc = gspread.authorize(creds)

print("✅ Google 授權成功")


✅ Google 授權成功


In [10]:
# Cell 5-2：開啟同一份試算表，把 full_df 寫到「整學期紀錄」分頁

sheet_url = "https://docs.google.com/spreadsheets/d/1lTEaj7A71Mf7VbPSyN0RPl1l-xIG2usKZJef35xxak0/edit?gid=0#gid=0"

sh = gc.open_by_url(sheet_url)
SHEET_NAME = "整學期紀錄"  # 你可以改成自己喜歡的名稱

# 如果已有同名分頁就清掉，沒有就新增
try:
    ws = sh.worksheet(SHEET_NAME)
    ws.clear()
    print(f"🔁 已找到工作表「{SHEET_NAME}」，清空舊資料。")
except gspread.WorksheetNotFound:
    ws = sh.add_worksheet(title=SHEET_NAME, rows="1000", cols="10")
    print(f"🆕 尚未存在，已新增工作表「{SHEET_NAME}」。")

# 寫入 DataFrame（會含欄位名稱）
set_with_dataframe(ws, full_df, include_index=False, include_column_header=True)

print(f"✅ 已將整學期課程紀錄寫入工作表「{SHEET_NAME}」")


🆕 尚未存在，已新增工作表「整學期紀錄」。
✅ 已將整學期課程紀錄寫入工作表「整學期紀錄」
